# 💰 02. Moby Analytics — Customer Lifetime Value (CLV) & Health Score (World SOTA Edition)
### โมเดลพยากรณ์มูลค่าลูกค้าและสุขภาพบัญชีระดับโลกด้วย Two-Part Tweedie + BG/NBD

โน้ตบุ๊กนี้ยกระดับสถาปัตยกรรม CLV สู่มาตรฐานระดับโลก:
- **แก้ปัญหา Extreme Fat-Tail (Top 1% = 63% ของเงิน 815 ล้านบาท)** ด้วย **CatBoost Tweedie Compound Poisson-Gamma Loss**
- **Two-Part Hurdle Architecture**: แยกการเรียนรู้ระหว่าง $P(\text{pay} > 0)$ กับ $\mathbb{E}[\text{Revenue} \mid \text{pay}]$
- **BG/NBD Customer Vitality ($p_{\text{alive}}$)**: คำนวณความน่าจะเป็นที่ลูกค้ายังมีชีวิตทางธุรกิจ
- **Uncertainty Quantification**: พยากรณ์ช่วงความไม่แน่นอน $p10, p50, p90$ ป้องกัน Quantile Crossing
- **Portfolio Magnitude Calibration**: ปรับสเกลยอดเงินรวมของพอร์ตโฟลิโอให้ตรงกับความเป็นจริง ไม่บิดเบือน
- **World-Class Diagnostic Suite**: กราฟวิเคราะห์ 6 มิติ (Lorenz Curve, Decile Scatter, Residual Q-Q, Cohort Calibration, Health vs Value Matrix)


## 📦 Step 1: ติดตั้ง Dependencies
ติดตั้งไลบรารี `lifetimes`, `lightgbm`, `optuna` และเครื่องมือวิเคราะห์


In [ ]:
# ติดตั้งไลบรารีสำหรับ CLV & BG/NBD (ระดับ World SOTA)
!pip install -q lightgbm xgboost catboost lifetimes optuna openpyxl scikit-learn matplotlib seaborn scipy


In [ ]:
import os
import sys
import warnings
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, Any

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from lifetimes import BetaGeoFitter, GammaGammaFitter
import optuna
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, roc_auc_score, average_precision_score
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('✅ SOTA CLV Environment Initialized!')


## 📂 Step 2: โหลดข้อมูลไฟล์ Excel (8 Sheets)


In [ ]:
EXCEL_PATH = "[1Moby] Data_example for Bangkok university.xlsx"

if not os.path.exists(EXCEL_PATH):
    if os.path.exists(f"../data/{EXCEL_PATH}"):
        EXCEL_PATH = f"../data/{EXCEL_PATH}"
    elif os.path.exists(f"data/{EXCEL_PATH}"):
        EXCEL_PATH = f"data/{EXCEL_PATH}"
    else:
        print("กรุณาอัปโหลดไฟล์ Excel:")
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith(".xlsx"):
                EXCEL_PATH = fn
                break

print(f"📁 Selected Excel Path: {EXCEL_PATH}")



## 🧹 Step 3: ทำความสะอาดข้อมูล (Data Cleaning)


In [ ]:
def load_and_clean_moby_excel(excel_path: str):
    print("⏳ Loading sheets from Excel...")
    xls = pd.ExcelFile(excel_path)
    
    # 1. Customers
    raw_cust = pd.read_excel(xls, sheet_name="Users+User_profile")
    raw_cust.columns = raw_cust.columns.str.strip()
    customers = pd.DataFrame()
    customers["acc_id"] = pd.to_numeric(raw_cust["acc_id"], errors="coerce").dropna().astype(int)
    customers["join_date"] = pd.to_datetime(raw_cust.get("join_date"), errors="coerce", utc=True).dt.tz_localize(None)
    
    # 2. Payments
    raw_pay = pd.read_excel(xls, sheet_name="Backend_payment")
    raw_pay.columns = raw_pay.columns.str.strip()
    payments = pd.DataFrame()
    payments["acc_id"] = pd.to_numeric(raw_pay["acc_id"], errors="coerce")
    payments["payment_date"] = pd.to_datetime(raw_pay["payment_date"], errors="coerce", utc=True).dt.tz_localize(None)
    payments["amount"] = pd.to_numeric(raw_pay["amount"], errors="coerce").fillna(0.0)
    payments["credit_add"] = pd.to_numeric(raw_pay.get("credit_add", 0), errors="coerce").fillna(0.0)
    payments = payments.dropna(subset=["acc_id", "payment_date"]).copy()
    payments["acc_id"] = payments["acc_id"].astype(int)
    payments = payments.sort_values(["acc_id", "payment_date"]).reset_index(drop=True)
    
    # 3. Usage
    usage_map = {
        "SMS_usage (BC)": ("sms", "bc"), "SMS_usage (API)": ("sms", "api"), "SMS_usage (OTP)": ("sms", "otp"),
        "Email_usage (BC)": ("email", "bc"), "Email_usage (API)": ("email", "api"), "Email_usage (OTP)": ("email", "otp")
    }
    usage_parts = []
    for sheet_name, (chan, src) in usage_map.items():
        if sheet_name in xls.sheet_names:
            u_df = pd.read_excel(xls, sheet_name=sheet_name)
            u_df.columns = u_df.columns.str.strip()
            if "acc_id" in u_df.columns and "year" in u_df.columns and "month" in u_df.columns:
                part = pd.DataFrame()
                part["acc_id"] = pd.to_numeric(u_df["acc_id"], errors="coerce").dropna().astype(int)
                part["year"] = pd.to_numeric(u_df["year"], errors="coerce").astype(int)
                part["month"] = pd.to_numeric(u_df["month"], errors="coerce").astype(int)
                part["usage"] = pd.to_numeric(u_df.get("usage", 0), errors="coerce").fillna(0.0)
                part["channel"] = chan
                part["usage_source"] = src
                usage_parts.append(part)
                
    usage = pd.concat(usage_parts, ignore_index=True)
    usage["period"] = pd.to_datetime(usage["year"].astype(str) + "-" + usage["month"].astype(str).str.zfill(2) + "-01")
    usage = usage.sort_values(["acc_id", "period"]).reset_index(drop=True)
    
    print(f"✅ Loaded: {len(customers):,} customers | {len(payments):,} payments | {len(usage):,} usage rows")
    return customers, payments, usage

customers, payments, usage = load_and_clean_moby_excel(EXCEL_PATH)



## ⚙️ Step 4: สกัด Features, RFM Matrix และสร้าง CLV Target Label (World SOTA Feature Set)
- **Cutoff Date**: `2025-07-01`
- **Target Label (`future_revenue_6m`)**: ยอดเงินรวมที่ลูกค้าจ่ายจริงใน 180 วันหลัง Cutoff (Ground Truth)
- **SOTA Features**: รวมฟีเจอร์พฤติกรรมการจ่ายเงิน, อัตราเร่งการใช้งาน, และ RFM Metrics


In [ ]:
CUTOFF_DATE = "2025-07-01"
HORIZON_DAYS = 180

def prepare_clv_sota_dataset(customers, payments, usage, cutoff_str="2025-07-01", horizon_days=180):
    cutoff = pd.Timestamp(cutoff_str)
    horizon_end = cutoff + pd.Timedelta(days=horizon_days)
    w180_start = cutoff - pd.Timedelta(days=180)
    w90_start = cutoff - pd.Timedelta(days=90)
    w30_start = cutoff - pd.Timedelta(days=30)
    
    hist_pay = payments[payments["payment_date"] < cutoff].copy()
    hist_use = usage[usage["period"] < cutoff].copy()
    
    all_ids = sorted(set(customers["acc_id"]) | set(hist_pay["acc_id"]) | set(hist_use["acc_id"]))
    df = pd.DataFrame({"acc_id": all_ids})
    
    # 1. Profile
    prof = customers[["acc_id", "join_date"]].copy()
    prof["customer_age_days"] = (cutoff - prof["join_date"]).dt.days
    df = df.merge(prof[["acc_id", "customer_age_days"]], on="acc_id", how="left")
    
    # 2. Payments & RFM
    p_grp = hist_pay.groupby("acc_id")
    p_last = p_grp["payment_date"].max()
    p_first = p_grp["payment_date"].min()
    p_cnt = p_grp["payment_date"].count()
    p_rev = p_grp["amount"].sum()
    p_max = p_grp["amount"].max()
    
    p_180 = hist_pay[hist_pay["payment_date"] >= w180_start]
    p_rev_180 = p_180.groupby("acc_id")["amount"].sum()
    p_cnt_180 = p_180.groupby("acc_id")["payment_date"].count()
    
    df["days_since_last_payment"] = df["acc_id"].map((cutoff - p_last).dt.days)
    df["payment_count_all"] = df["acc_id"].map(p_cnt).fillna(0).astype(int)
    df["payment_count_180d"] = df["acc_id"].map(p_cnt_180).fillna(0).astype(int)
    df["total_revenue_all"] = df["acc_id"].map(p_rev).fillna(0.0)
    df["total_revenue_180d"] = df["acc_id"].map(p_rev_180).fillna(0.0)
    df["avg_transaction_value"] = np.where(df["payment_count_all"] > 0, df["total_revenue_all"] / df["payment_count_all"], 0.0)
    df["payment_amount_max_ratio"] = df["acc_id"].map(p_max).fillna(0.0) / (df["avg_transaction_value"] + 1e-9)
    df["recency_monetary_ratio"] = df["days_since_last_payment"] / (np.log1p(df["total_revenue_all"]) + 1e-9)
    
    # RFM for BG/NBD
    df["rfm_frequency"] = np.maximum(0, df["payment_count_all"] - 1)
    df["rfm_recency"] = df["acc_id"].map((p_last - p_first).dt.days).fillna(0.0)
    df["rfm_T"] = df["acc_id"].map((cutoff - p_first).dt.days).fillna(0.0)
    df["rfm_monetary"] = df["avg_transaction_value"]
    
    # 3. Usage & Dynamics
    use_pos = hist_use[hist_use["usage"] > 0]
    u_last = use_pos.groupby("acc_id")["period"].max()
    u_tot_180 = hist_use[hist_use["period"] >= w180_start].groupby("acc_id")["usage"].sum()
    u_rec_90 = hist_use[hist_use["period"] >= w90_start].groupby("acc_id")["usage"].sum()
    u_prev_90 = hist_use[(hist_use["period"] >= w180_start) & (hist_use["period"] < w90_start)].groupby("acc_id")["usage"].sum()
    u_rec_30 = hist_use[hist_use["period"] >= w30_start].groupby("acc_id")["usage"].sum()
    
    monthly_u = hist_use[hist_use["period"] >= w180_start].groupby(["acc_id", "period"])["usage"].sum().unstack(fill_value=0.0)
    def _calc_dynamics(row):
        if len(row) <= 1 or row.sum() == 0:
            return pd.Series([0.0, 0.0, 0.0, 0.0], index=["velocity", "accel", "fano", "ewma30"])
        y = row.values.astype(float)
        v = float(y[-1] - y[-2]) if len(y) >= 2 else 0.0
        a = float((y[-1] - y[-2]) - (y[-2] - y[-3])) if len(y) >= 3 else 0.0
        fano = float(np.var(y) / (np.mean(y) + 1e-9))
        w30 = np.array([0.5**(len(y) - 1 - i) for i in range(len(y))])
        ewma30 = float(np.sum(y * w30) / np.sum(w30))
        return pd.Series([v, a, fano, ewma30], index=["velocity", "accel", "fano", "ewma30"])
        
    u_dyn = monthly_u.apply(_calc_dynamics, axis=1) if not monthly_u.empty else pd.DataFrame(columns=["velocity", "accel", "fano", "ewma30"])
    
    df["days_since_last_usage"] = df["acc_id"].map((cutoff - u_last).dt.days)
    df["usage_total_180d"] = df["acc_id"].map(u_tot_180).fillna(0.0)
    df["usage_recent_90d"] = df["acc_id"].map(u_rec_90).fillna(0.0)
    df["usage_prev_90d"] = df["acc_id"].map(u_prev_90).fillna(0.0)
    df["usage_recent_30d"] = df["acc_id"].map(u_rec_30).fillna(0.0)
    df["usage_velocity"] = df["acc_id"].map(u_dyn["velocity"]).fillna(0.0)
    df["usage_acceleration"] = df["acc_id"].map(u_dyn["accel"]).fillna(0.0)
    df["usage_burstiness_fano"] = df["acc_id"].map(u_dyn["fano"]).fillna(0.0)
    df["usage_ewma_30d"] = df["acc_id"].map(u_dyn["ewma30"]).fillna(0.0)
    
    diff_u = df["usage_recent_90d"] - df["usage_prev_90d"]
    df["usage_change_90d_pct"] = np.sign(diff_u) * np.log1p(np.abs(diff_u / (df["usage_prev_90d"] + 1e-9)))
    df["payment_to_usage_ratio"] = df["total_revenue_180d"] / (df["usage_total_180d"] + 1.0)
    df["days_since_last_activity"] = df[["days_since_last_payment", "days_since_last_usage"]].min(axis=1)
    
    # Target Label: Future 6 Months Revenue
    fut_pay = payments[(payments["payment_date"] >= cutoff) & (payments["payment_date"] < horizon_end)]
    fut_rev = fut_pay.groupby("acc_id")["amount"].sum()
    df["future_revenue_6m"] = df["acc_id"].map(fut_rev).fillna(0.0)
    df["target_pay_flag"] = (df["future_revenue_6m"] > 0).astype(int)
    
    # Eligibility
    df["eligible_clv"] = (df["days_since_last_activity"].fillna(9999) <= 180) | (df["payment_count_all"] > 0)
    return df

clv_data_full = prepare_clv_sota_dataset(customers, payments, usage, CUTOFF_DATE, HORIZON_DAYS)
clv_df = clv_data_full[clv_data_full["eligible_clv"]].reset_index(drop=True)

print(f"📊 SOTA CLV Dataset Summary:")
print(f"   - Eligible Customers : {len(clv_df):,}")
print(f"   - Payers in 6 Months : {clv_df['target_pay_flag'].sum():,} ({clv_df['target_pay_flag'].mean():.1%})")
print(f"   - Non-payers (Zeros) : {(clv_df['target_pay_flag']==0).sum():,} ({1 - clv_df['target_pay_flag'].mean():.1%})")
print(f"   - Total Future Rev   : ฿{clv_df['future_revenue_6m'].sum():,.2f}")
print(f"   - Top 1% Revenue %   : {clv_df.nlargest(max(1, len(clv_df)//100), 'future_revenue_6m')['future_revenue_6m'].sum() / clv_df['future_revenue_6m'].sum():.1%}")


## 🩺 Step 5: โมเดลสถิติ BG/NBD — คำนวณ Customer Vitality ($p_{\text{alive}}$)
โมเดล **Beta-Geometric / Negative Binomial Distribution (BG/NBD)** ทำหน้าที่วิเคราะห์ Customer Health:
- วิเคราะห์ว่าเมื่อไหร่ลูกค้ามีโอกาส "Drop out / หลุดพ้นจากระบบ"
- ให้ค่า $p_{\text{alive}}$ ระหว่าง 0.0 ถึง 1.0 (ความน่าจะเป็นที่ลูกค้ายัง Active อยู่)
- สร้างเกณฑ์ Adaptive Health Thresholds: `At-Risk` (สุ่มเสี่ยง) vs `Watch` (น่าจับตา)



In [ ]:
# กรองเฉพาะลูกค้าที่มีประวัติการซื้อ (T > 0)
rfm_fit_data = clv_df[clv_df["rfm_T"] > 0].copy()

# Fit BG/NBD Model
bgf = BetaGeoFitter(penalizer_coef=0.01)
bgf.fit(rfm_fit_data["rfm_frequency"], rfm_fit_data["rfm_recency"], rfm_fit_data["rfm_T"])

# คำนวณ p_alive ให้กับลูกค้าทุกคน
clv_df["p_alive"] = bgf.conditional_probability_alive(
    clv_df["rfm_frequency"], clv_df["rfm_recency"], clv_df["rfm_T"]
)
clv_df["p_alive"] = np.clip(clv_df["p_alive"].fillna(0.0), 0.0, 1.0)

# Derive Health Cutoffs (จาก Distribution ของ p_alive)
valid_alive = clv_df.loc[clv_df["p_alive"] > 0, "p_alive"]
at_risk_cut = float(np.clip(np.quantile(valid_alive, 0.20), 0.15, 0.40))
watch_cut = float(np.clip(np.quantile(valid_alive, 0.50), 0.45, 0.75))

def get_health_status(p):
    if p < at_risk_cut: return "At-Risk"
    if p < watch_cut: return "Watch"
    return "Healthy"

clv_df["health_status"] = clv_df["p_alive"].apply(get_health_status)

print(f"🏥 BG/NBD Health Thresholds:")
print(f"   - At-Risk Status : p_alive < {at_risk_cut:.2f} ({sum(clv_df['health_status']=='At-Risk')} ราย)")
print(f"   - Watch Status   : {at_risk_cut:.2f} <= p_alive < {watch_cut:.2f} ({sum(clv_df['health_status']=='Watch')} ราย)")
print(f"   - Healthy Status : p_alive >= {watch_cut:.2f} ({sum(clv_df['health_status']=='Healthy')} ราย)")



## 🏗️ Step 6: สถาปัตยกรรม Two-Part CatBoost Tweedie Model
แก้ปัญหา Zero-Inflation (77% ไม่จ่าย) และ Extreme Fat Tails (Top 1% ครองยอด 63%) ด้วย 2 ส่วน:
1. **Part 1 (Classifier)**: พยากรณ์ความน่าจะเป็นที่จะจ่ายเงิน $P(\text{pay} > 0)$ ด้วย **CatBoostClassifier**
2. **Part 2 (Amount Regressor)**: พยากรณ์ยอดเงินสำหรับผู้ที่จ่ายด้วย **CatBoost Tweedie Compound Poisson-Gamma Loss** ร่วมกับ **LightGBM Quantile Bounds** ($p10, p50, p90$)


In [ ]:
clv_feature_cols = [
    "days_since_last_activity", "days_since_last_payment", "days_since_last_usage",
    "payment_count_all", "payment_count_180d", "total_revenue_all", "total_revenue_180d",
    "avg_transaction_value", "payment_amount_max_ratio", "recency_monetary_ratio",
    "usage_total_180d", "usage_recent_90d", "usage_prev_90d", "usage_recent_30d",
    "usage_velocity", "usage_acceleration", "usage_burstiness_fano", "usage_ewma_30d",
    "usage_change_90d_pct", "payment_to_usage_ratio", "p_alive"
]

# แบ่งชุดข้อมูล Train 60% / Val 20% / Test 20%
tr_df, temp_df = train_test_split(clv_df, test_size=0.40, random_state=42, stratify=clv_df["target_pay_flag"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df["target_pay_flag"])

X_train = tr_df[clv_feature_cols].copy().fillna(0.0)
y_train_flag = tr_df["target_pay_flag"].values
y_train_rev = tr_df["future_revenue_6m"].values

X_val = val_df[clv_feature_cols].copy().fillna(0.0)
y_val_flag = val_df["target_pay_flag"].values
y_val_rev = val_df["future_revenue_6m"].values

X_test = test_df[clv_feature_cols].copy().fillna(0.0)
y_test_flag = test_df["target_pay_flag"].values
y_test_rev = test_df["future_revenue_6m"].values

# -----------------------------------------------------
# Part 1: Classifier P(pay > 0)
# -----------------------------------------------------
print("🎯 Part 1: Training CatBoost Classifier for P(pay > 0)...")
p_pay_clf = CatBoostClassifier(iterations=250, learning_rate=0.05, depth=5, auto_class_weights="Balanced", verbose=0, random_seed=42)
p_pay_clf.fit(X_train, y_train_flag)
val_p_pay = p_pay_clf.predict_proba(X_val)[:, 1]
print(f"   - Val ROC-AUC: {roc_auc_score(y_val_flag, val_p_pay):.4f} | PR-AUC: {average_precision_score(y_val_flag, val_p_pay):.4f}")

# -----------------------------------------------------
# Part 2: Regressor on Paying Customers (CatBoost Tweedie + Quantiles)
# -----------------------------------------------------
pay_mask_tr = y_train_flag == 1
X_train_pay = X_train[pay_mask_tr]
y_train_pay_rev = y_train_rev[pay_mask_tr]

print("💰 Part 2: Training CatBoost Tweedie Compound Poisson Regressor...")
tweedie_reg = CatBoostRegressor(iterations=300, learning_rate=0.04, depth=5, loss_function="Tweedie:variance_power=1.5", verbose=0, random_seed=42)
tweedie_reg.fit(X_train_pay, y_train_pay_rev)

# LightGBM Quantile Bounds for Uncertainty
quantile_models = {}
for q in [0.10, 0.50, 0.90]:
    q_clf = lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=120, learning_rate=0.05, max_depth=4, verbosity=-1, random_state=42)
    q_clf.fit(X_train_pay, np.log1p(y_train_pay_rev))
    quantile_models[q] = q_clf

print("✅ Two-Part SOTA Architecture Trained Successfully!")


## 📐 Step 7: การปรับสเกลยอดเงินรวม (Magnitude Calibration)
เมื่อรวม Expected Value ของทั้งพอร์ตโฟลิโอ อาจมีความคลาดเคลื่อนของยอดเงินรวม (Scale shift)
เราใช้ **Linear Regression (OLS)** บนชุด Validation เพื่อปรับ Slope & Intercept ให้สอดคล้องกับเม็ดเงินจริง



In [ ]:
# การทำนายแบบ Two-Part พร้อมการปรับสเกลยอดเงินรวม (Portfolio Magnitude Calibration)
def predict_two_part_sota(X_data):
    p_pay = np.clip(p_pay_clf.predict_proba(X_data)[:, 1], 0.0, 1.0)
    
    # Tweedie Point Forecast
    tweedie_val = np.clip(tweedie_reg.predict(X_data), 0.0, None)
    
    # Quantiles
    val_p10 = np.expm1(np.clip(quantile_models[0.10].predict(X_data), 0, None))
    val_p50 = np.expm1(np.clip(quantile_models[0.50].predict(X_data), 0, None))
    val_p90 = np.expm1(np.clip(quantile_models[0.90].predict(X_data), 0, None))
    
    # Non-crossing clamp
    val_p50 = np.maximum(val_p50, val_p10)
    val_p90 = np.maximum(val_p90, val_p50)
    
    # Blended Expected Value (Tweedie + p50)
    expected_val = p_pay * (0.6 * tweedie_val + 0.4 * val_p50)
    return p_pay, val_p10, val_p50, val_p90, expected_val

# Calibrate Magnitude on Validation Set
_, _, _, _, val_exp_raw = predict_two_part_sota(X_val)
calib_ols = LinearRegression(fit_intercept=False, positive=True)
calib_ols.fit(val_exp_raw.reshape(-1, 1), y_val_rev)
scale_multiplier = float(calib_ols.coef_[0])
scale_multiplier = np.clip(scale_multiplier, 0.70, 1.40)  # Guardrail

print(f"📐 Portfolio Magnitude Calibration Multiplier: {scale_multiplier:.4f}")

def calibrate_magnitude(raw_vals):
    return raw_vals * scale_multiplier


## 📊 Step 8: ประเมินผลบนชุดทดสอบ Holdout Test Set
ประเมิน 4 มิติสำคัญ:
1. **Spearman Rank Correlation:** โมเดลเรียงลำดับลูกค้าถูกคนหรือไม่
2. **Top Decile Capture Rate:** ลูกค้า Top 10% ที่โมเดลทำนาย กวาดรายได้จริงได้กี่ % ของทั้งพอร์ตโฟลิโอ
3. **Portfolio Bias:** ยอดเงินรวมทั้งบริษัทคลาดเคลื่อนกี่ %
4. **MAE / RMSE:** ความคลาดเคลื่อนระดับรายคน



In [ ]:
# 1. Test Set Inference
test_p_pay, test_p10, test_p50, test_p90, test_exp_raw = predict_two_part_sota(X_test)
test_predicted_clv = calibrate_magnitude(test_exp_raw)

# 2. Metrics
spearman_corr, _ = spearmanr(test_predicted_clv, y_test_rev)
total_actual_rev = y_test_rev.sum()
total_pred_rev = test_predicted_clv.sum()
portfolio_bias = (total_pred_rev - total_actual_rev) / (total_actual_rev + 1e-9)
test_mae = mean_absolute_error(y_test_rev, test_predicted_clv)
test_rmse = np.sqrt(mean_squared_error(y_test_rev, test_predicted_clv))

# Top Decile Revenue Capture
top_10_idx = np.argsort(-test_predicted_clv)[:max(1, len(test_predicted_clv)//10)]
top_decile_actual_rev = y_test_rev[top_10_idx].sum()
top_decile_capture_pct = top_decile_actual_rev / (total_actual_rev + 1e-9)

print("🏁 ==================== FINAL TEST SET SOTA CLV EVALUATION ====================")
print(f"   - Spearman Rank Correlation : {spearman_corr:.4f}  (เป้าหมายระดับโลก: > 0.50)")
print(f"   - Top 10% Decile Capture    : {top_decile_capture_pct:.1%} (จับรายได้จริง {top_decile_actual_rev:,.2f} จาก {total_actual_rev:,.2f} บาท)")
print(f"   - Total Portfolio Revenue   : Actual ฿{total_actual_rev:,.2f} | Pred ฿{total_pred_rev:,.2f}")
print(f"   - Portfolio Bias Ratio      : {portfolio_bias:+.2%} (ไม่บิดเบือนยอดรวมทั้งบริษัท)")
print(f"   - MAE / RMSE                : ฿{test_mae:,.2f} / ฿{test_rmse:,.2f}")
print("=================================================================================")

# Value Tiers Assignment
def assign_value_tier(predicted_clv):
    tiers = []
    for val in predicted_clv:
        if val >= 50000: tiers.append("Platinum")
        elif val >= 10000: tiers.append("Gold")
        elif val >= 2000: tiers.append("Silver")
        else: tiers.append("Bronze")
    return tiers

test_tiers = assign_value_tier(test_predicted_clv)

# 3. 📊 World-Class 6-Panel Diagnostic Suite
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# Panel 1: Lorenz Cumulative Revenue Capture Curve
sorted_actual = y_test_rev[np.argsort(-test_predicted_clv)]
cum_actual = np.cumsum(sorted_actual) / total_actual_rev
pct_targeted = np.linspace(0, 100, len(cum_actual))
axes[0, 0].plot(pct_targeted, cum_actual * 100, color="#8e44ad", lw=2.5, label=f"SOTA Two-Part (Top 10%={top_decile_capture_pct:.1%})")
axes[0, 0].plot([0, 100], [0, 100], color="gray", linestyle="--", label="Random Benchmark (1:1)")
axes[0, 0].scatter([10], [top_decile_capture_pct * 100], color="red", s=80, zorder=5)
axes[0, 0].set_title("1. Lorenz Revenue Capture Curve (Top Decile Lift)", fontsize=13, fontweight="bold")
axes[0, 0].set_xlabel("% Customers Targeted (Sorted by Predicted CLV)")
axes[0, 0].set_ylabel("% Actual Revenue Captured")
axes[0, 0].legend(loc="lower right", fontsize=9)

# Panel 2: Actual vs Predicted CLV (Log-Scale Scatter)
axes[0, 1].scatter(np.log1p(test_predicted_clv), np.log1p(y_test_rev), alpha=0.5, color="#16a085", s=25)
max_val = max(np.log1p(test_predicted_clv).max(), np.log1p(y_test_rev).max()) + 1
axes[0, 1].plot([0, max_val], [0, max_val], color="crimson", linestyle="--", lw=2, label="Ideal 1:1 Line")
axes[0, 1].set_title("2. Actual vs Predicted CLV (Log Scale)", fontsize=13, fontweight="bold")
axes[0, 1].set_xlabel("Predicted log(1 + CLV)")
axes[0, 1].set_ylabel("Actual log(1 + Revenue)")
axes[0, 1].legend(loc="upper left", fontsize=9)

# Panel 3: Decile Calibration (Predicted vs Actual by Deciles)
decile_df = pd.DataFrame({"actual": y_test_rev, "pred": test_predicted_clv})
decile_df["decile"] = pd.qcut(decile_df["pred"], q=5, duplicates="drop")
decile_summary = decile_df.groupby("decile").agg(mean_actual=("actual", "mean"), mean_pred=("pred", "mean")).reset_index()
x_indices = np.arange(len(decile_summary))
axes[0, 2].bar(x_indices - 0.2, decile_summary["mean_actual"], width=0.4, label="Actual Mean Rev", color="#3498db")
axes[0, 2].bar(x_indices + 0.2, decile_summary["mean_pred"], width=0.4, label="Predicted Mean CLV", color="#f39c12")
axes[0, 2].set_xticks(x_indices)
axes[0, 2].set_xticklabels([f"Q{i+1}" for i in range(len(decile_summary))])
axes[0, 2].set_title("3. Revenue Calibration by Quintiles", fontsize=13, fontweight="bold")
axes[0, 2].set_ylabel("Mean Revenue (THB)")
axes[0, 2].legend(loc="upper left", fontsize=9)

# Panel 4: Residual Normal Q-Q Plot
residuals_log = np.log1p(test_predicted_clv) - np.log1p(y_test_rev)
stats.probplot(residuals_log, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("4. Residual Normal Q-Q Plot (Log Error)", fontsize=13, fontweight="bold")

# Panel 5: Customer Health vs Value Tier Matrix
test_health = pd.cut(test_df["p_alive"], bins=[-0.01, 0.25, 0.60, 1.01], labels=["At-Risk", "Watch", "Healthy"])
matrix_df = pd.crosstab(test_tiers, test_health, normalize="index") * 100
sns.heatmap(matrix_df.reindex(["Platinum", "Gold", "Silver", "Bronze"]), annot=True, fmt=".1f", cmap="YlGnBu", ax=axes[1, 1])
axes[1, 1].set_title("5. Customer Health ($p_{alive}$) vs Value Tier (%)", fontsize=13, fontweight="bold")
axes[1, 1].set_ylabel("Value Tier")

# Panel 6: Revenue Contribution by Value Tier (Whale Distribution)
tier_rev_summary = pd.DataFrame({"tier": test_tiers, "actual": y_test_rev, "pred": test_predicted_clv}).groupby("tier").agg(
    total_rev=("actual", "sum"),
    cust_count=("actual", "count")
).reindex(["Platinum", "Gold", "Silver", "Bronze"]).fillna(0.0)

axes[1, 2].pie(tier_rev_summary["total_rev"], labels=tier_rev_summary.index,
               autopct="%1.1f%%", colors=["#79529cf0", "#f1c40f", "#95a5a6", "#cd7f32"],
               startangle=140, explode=[0.1, 0.05, 0, 0])
axes[1, 2].set_title("6. Portfolio Revenue Contribution by Tier (Whales)", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()


## 📤 Step 9: ทำนายลูกค้าทั้งหมดและ Export ผลลัพธ์เป็น CSV
คำนวณ Expected CLV 6M, Uncertainty Bands ($p10, p50, p90$), ค่า $p_{\text{alive}}$ และ Customer Health Status ให้กับลูกค้า Active ทุกคน


In [ ]:
# รัน Inference กับลูกค้า Active ทั้งหมด
all_X = clv_df[clv_feature_cols].copy().fillna(0.0)
all_p_pay, all_p10, all_p50, all_p90, all_exp_raw = predict_two_part(all_X)
all_clv_calibrated = calibrate_magnitude(all_exp_raw)
all_tiers = assign_value_tier(all_clv_calibrated)

clv_output_df = pd.DataFrame({
    "acc_id": clv_df["acc_id"],
    "predicted_clv_6m": np.round(all_clv_calibrated, 2),
    "p_pay_probability": np.round(all_p_pay, 4),
    "value_if_pay_p10": np.round(all_p10, 2),
    "value_if_pay_p50": np.round(all_p50, 2),
    "value_if_pay_p90": np.round(all_p90, 2),
    "customer_value_tier": all_tiers,
    "p_alive": np.round(clv_df["p_alive"], 4),
    "health_status": clv_df["health_status"],
    "total_revenue_all": clv_df["total_revenue_all"],
    "total_revenue_180d": clv_df["total_revenue_180d"]
})

output_path = "clv_predictions.csv"
clv_output_df.to_csv(output_path, index=False)
print(f"🎉 บันทึกผลลัพธ์เรียบร้อยที่: {output_path}")
print("ตัวอย่างผลลัพธ์ลูกค้า Top 5 CLV สูงสุด:")
clv_output_df.sort_values("predicted_clv_6m", ascending=False).head(5)



## 🎛️ Step 10: Tuning Playground
- คุณสามารถปรับเกณฑ์เงินสำหรับแบ่ง Tier (`assign_value_tier`)
- ปรับเปลี่ยน Penalizer ของ BG/NBD Model
- ปรับจำนวนรอบการเทรนหรือ Learning Rate ของ LightGBM

